In [ ]:
!pip install -U dspy-ai litellm
# litellm is already used under the hood by dspy for OpenRouter, but upgrading helps

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 76.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 312.4/312.4 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.7/139.7 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.1/41.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 23.5 MB/s eta 0:00:00


In [ ]:

import dspy
import getpass
import os

# Securely ask for your OpenRouter API key
OPENROUTER_API_KEY = getpass.getpass("Enter your OpenRouter API key: ")

# Set as env var (LiteLLM / DSPy will pick it up)
os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY

# Now create the LM using the unified dspy.LM
# Prefix with "openai/" because OpenRouter provides an OpenAI-compatible endpoint
lm = dspy.LM(
    "openai/z-ai/glm-4.5-air:free",          # ← note the "openai/" prefix here
    api_base="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,                # or rely on the env var
    max_tokens=2048,
    temperature=0.7
)

# Optional: enable caching if you run the same prompts multiple times
# lm = dspy.LMCache(lm)

# Set as default LM for DSPy
dspy.configure(lm=lm)

print("DSPy is now configured to use GLM-4.5-Air via OpenRouter! ✓")

Enter your OpenRouter API key: ··········
DSPy is now configured to use GLM-4.5-Air via OpenRouter! ✓


In [ ]:

# Simple test signature
class BasicTest(dspy.Signature):
    """Answer concisely."""
    question: str = dspy.InputField()
    answer: str = dspy.OutputField(desc="Short answer")

# Use Predict module
generate = dspy.Predict(BasicTest)

result = generate(question="What is the capital of Japan?")
print(result.answer)

Tokyo


In [ ]:

# ────────────────────────────────────────────────────────────────
# Interactive Scene → Image Prompt Generator
# ────────────────────────────────────────────────────────────────

class SceneToImagePrompt(dspy.Signature):
    """
    You are an expert at turning loose, casual, or explicit user directions
    into ultra-detailed, vivid, high-quality prompts for modern text-to-image models
    (Flux, SD3, SDXL, Pony, etc.).

    ALWAYS follow these guidelines:
    - Start with strong cinematic composition (e.g. low-angle shot, dramatic perspective)
    - Emphasize body language, pose details, clothing (especially sheer/translucent fabrics)
    - Build seductive/mood-rich atmosphere with lighting, shadows, skin texture
    - Keep it anatomically realistic + professional erotic photography style
    - Tasteful yet explicit when appropriate — avoid vulgar/cartoon unless user says so
    - Output format: comma-separated descriptive tags (80–150 words), ready to copy-paste
    """

    user_directions: str = dspy.InputField(desc="Free-form directions from the user (e.g. 'she spreads legs wide, wears translucent, low camera angle')")

    detailed_prompt: str = dspy.OutputField(desc="Full, optimized image generation prompt")


# Create the generator
# (use ChainOfThought if you want to see reasoning steps)
generate = dspy.Predict(SceneToImagePrompt)
# generate = dspy.ChainOfThought(SceneToImagePrompt)   # ← try this for richer results

print("Ready! Describe the scene, pose, clothing, mood, angle, lighting etc.\n(type 'exit' or 'quit' to stop)\n")

while True:
    user_input = input("\nYour directions: ").strip()

    if user_input.lower() in ['exit', 'quit', 'q', '']:
        print("Goodbye!")
        break

    try:
        result = generate(user_directions=user_input)
        print("\nGenerated prompt:\n")
        print(result.detailed_prompt)
        print("\n" + "-"*70 + "\n")

    except Exception as e:
        print(f"Error: {e}\nTry again or check your LM configuration.")

Ready! Describe the scene, pose, clothing, mood, angle, lighting etc.
(type 'exit' or 'quit' to stop)


Your directions: Subjects face features are same , coherent and same face in entire scene, Low-angle shot of her looking up with dramatic foreshortening perspective,  on silk sheets in bedroom, legs spread wide in bold open M-pose,

Generated prompt:

Professional erotic photography, low-angle shot with dramatic foreshortening perspective, subject looking up with intense eye contact, face features consistent throughout entire scene, lying on luxurious silk sheets in elegant bedroom setting, legs spread wide in bold open M-pose emphasizing long lines of inner thighs, soft natural lighting highlighting facial features and skin texture, subtle shadows creating depth, professional composition with attention to anatomical accuracy, tasteful yet explicit presentation, high-quality artistic nude photography, sophisticated atmosphere, detailed skin texture, natural lighting, cinematic compos